# Requirements

In [1]:
# https://www.tensorflow.org/api_docs/python/tf/keras/layers

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Get Data

In [2]:
# transformando e abrindo o arquivo em formato csv
df = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

# Data Prep Functions

* **preprocess_data**: selects the dataset columns that are going to be used.
* **preprocess_text**:
* **tokenizacao**:
* **treinar_vetorizacao**:
* **vectorize2**:

In [3]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    
    return df

def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df


def treinar_vetorizacao(df):

    # coleciona as palavras usadas para o treinamento da função CountVectorizer
    lista_treino = []

    for item in df['token']:
        lista_treino1 = [n for n in item if n not in lista_treino]
        lista_treino.extend(lista_treino1)

    # treina o modelo de vetorização
    vectorize = CountVectorizer(lowercase=True, strip_accents = 'unicode')

    vectorize.fit(lista_treino)
    
    return vectorize, lista_treino


def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2

# Pipeline Data Prep

In [4]:
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

#
df2 = preprocess_data(df, 
                      columns=['KEY','COMMENT_ID','COMMENT_TEXT'],
                      null = True
                     )

#
df2 = tokenizacao(df2)

#
vectorize, lista_treino = treinar_vetorizacao(df2)

# creates the dataset column with the vectorized text
df2['vectors'] = df2['token'].apply(vectorize2)

# creates an intermediary dataframe
df3 = df2.copy()

# 
for n, item in enumerate(df2.token):
    if len(item)==0:
        df3 = df3.drop(n)

# Section Name

In [9]:
topics = ['AQUECIMENTO', 
          'ASSISTÊNCIA TÉCNICA', 
          'ATENDIMENTO', 
          'AUTO FALANTE', 
          'BATERIA', 
          'CAMERA', 
          'CARREGADOR', 
          'CUSTO BENEFICIO', 
          'DESIGN', 
          'ENTREGA', 
          'FLASH', 
          'FONE', 
          'JOGOS', 
          'MEMÓRIA',
          'PESO', 
          'PREÇO', 
          'PROCESSADOR', 
          'QUALIDADE', 
          'RESISTÊNCIA', 
          'TAMANHO', 
          'TELA', 
          'TRAVAMENTO',
          'VELOCIDADE', 
          'GENÉRICO/OUTRO'
         ]

dic_topics = {}
for n, item in enumerate(topics):
    dic_topics[n]=item

topics_token = [preprocess_text(item) for item in topics]

# Section Name

In [11]:
# defines empty list
lista_clusters = []

#
for n, objeto in enumerate(topics_token):
    if len(objeto) == 1:
        lista_1 = [item for item in df3.token if objeto[0] in item]
        for n, item in enumerate(lista_clusters):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_clusters.append(lista_1)

    else:
        lista_2 = [item for item in df3.token if (
            objeto[0] in item) and (objeto[1] in item)]
        for n, item in enumerate(lista_clusters):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_clusters.append(lista_2)

# Section Name

In [13]:
temp = []

for n, objeto in enumerate(lista_clusters):
    temp.extend(objeto)

ultimo_cluster = [item for item in df3.token if item not in temp]

In [14]:
lista_clusters[23] = ultimo_cluster

In [15]:
print([len(item) for item in lista_clusters])

[31, 54, 427, 27, 6255, 2759, 991, 2180, 207, 2546, 16, 310, 136, 346, 11, 1131, 104, 1806, 11, 161, 507, 6, 20, 42024]


In [16]:
df3.shape, sum([len(item) for item in lista_clusters])

((62066, 5), 62066)

In [17]:
lista_clusters1 = lista_clusters.copy()

In [18]:
for n, item in enumerate(lista_clusters1):
    if len(item) > 3500:
        lista_clusters1[n]=random.sample(item, 2000)    

In [19]:
a = [len(item) for item in lista_clusters1]

print(a)

[31, 54, 427, 27, 2000, 2759, 991, 2180, 207, 2546, 16, 310, 136, 346, 11, 1131, 104, 1806, 11, 161, 507, 6, 20, 2000]


In [20]:
sum(a)

17787

# Vectorization

In [21]:
nlp = spacy.load('pt_core_news_md')
#nlp = spacy.load('pt_core_news_sm')

def vec(s):
    return nlp.vocab[s].vector

# Section Name

In [26]:
vec_size = 300
rows = sum(a)

list_of_matrix = [] 

final_feature_matrix = np.empty([rows, vec_size])

for n, item in enumerate(lista_clusters1):
    for corpus in item: 
        matrix = np.empty([len(corpus), vec_size]) 
                                              
        for idx, word in enumerate(corpus):
            matrix[idx,:] = vec(word) 
        list_of_matrix.append(matrix)



for row in range(rows):
    final_feature_matrix[row,:] = list_of_matrix[row].mean(axis = 0)

In [ ]:
final_feature_matrix.shape, sum(a)

In [22]:
labels = []

for n, item in enumerate(lista_clusters1):
    for i, objeto in enumerate(item):
        labels.append(n)

In [ ]:
labels.count(3)

In [23]:
x = np.array(labels)

In [24]:
x = x.reshape(-1,1)

In [ ]:
final_matrix = np.concatenate((final_feature_matrix, x), axis=1)

final_matrix.shape

In [ ]:
final_matrix[:,-1].reshape(-1,1)

In [ ]:
final_matrix1 = shuffle(final_matrix, random_state=42)

In [ ]:
final_matrix1.shape

In [ ]:
Xtreino, Xteste, ytreino, yteste = train_test_split(
    final_matrix1[:, 0:-1], final_matrix1[:, -1], train_size = 0.7, random_state=42)

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 2)

knn.fit(Xtreino, ytreino)

In [ ]:
pred = knn.predict(Xteste)

In [ ]:
accuracy_score(yteste,pred)

In [ ]:
a = 'Otimo'

a

In [ ]:
textoProcessado = preprocess_text(a)
matrix = np.empty([len(textoProcessado), 300])

In [ ]:
for idx, word in enumerate(textoProcessado):
    matrix[idx,:] = vec(word)

#
final_feature_matrix = np.empty([1, 300])

#
final_feature_matrix = matrix.mean(axis = 0).reshape(1,-1)

In [ ]:
final_feature_matrix.shape

In [ ]:
knn.predict(final_feature_matrix)

In [ ]:
knn.predict(final_feature_matrix)[0]

In [ ]:
dic_topics[knn.predict(final_feature_matrix)[0]]

In [ ]:
#def classifi():
#    texto = input("Digite o texto no qual você quer classificar aqui: \n\n")
#    textoProcessado = preprocess_text(texto)
#    matrix = np.empty([len(textoProcessado), 300])
#    for idx, word in enumerate(textoProcessado):
#         matrix[idx,:] = vec(word)
#    final_feature_matrix = np.empty([1, 300])
#    final_feature_matrix = matrix.mean(axis = 0).reshape(1,-1)